<a href="https://colab.research.google.com/github/akbar260/resumeMatcher/blob/main/QuoteMatcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**step 1: installing and importing libraries**

In [1]:
!pip install -q datasets sentence-transformers scikit-learn pandas numpy

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

**step 2: loading and cleaning the given dataset in the task**

In [3]:
quotes_data = load_dataset("Abirate/english_quotes", split="train")
df = quotes_data.to_pandas()

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [5]:
# Remove rows where the quote text is missing
df = df.dropna(subset=["quote"])

# Remove exact duplicate quotes
df = df.drop_duplicates(subset=["quote"])

# Reset the row numbers so they go 0, 1, 2, 3... after the rows we removed
df = df.reset_index(drop=True)

**step 3: for making it easier for the AI model we are combining quotes , author and tags into a single sentence.**

In [6]:
# This function turns a list of tags like ['life', 'love'] into the text "life, love"
def tags_to_text(tag_list):
    if tag_list is None or len(tag_list) == 0:
        return ""
    return ", ".join(tag_list)

# Apply that function to every row
df["tags_text"] = df["tags"].apply(tags_to_text)

# Now build one combined text field per quote, e.g.:
# "Be yourself... — Oscar Wilde. Topics: individuality, love"
df["combined_text"] = df["quote"] + " — " + df["author"].fillna("Unknown") + ". Topics: " + df["tags_text"]

**step 4: Loading the AI models**

In [7]:
model_names = {
    "MiniLM": "all-MiniLM-L6-v2",
    "MPNet": "all-mpnet-base-v2",
    "BGE-small": "BAAI/bge-small-en-v1.5",
    "E5-small": "intfloat/e5-small-v2",
}

# Load each model and store it in another dictionary, keyed by our nickname
models = {}
for nickname, real_name in model_names.items():
    print(f"Loading {nickname}...")
    models[nickname] = SentenceTransformer(real_name)

Loading MiniLM...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

**step 5: not necessary but for BGE-small and E5-small to work efficiently we need to add text before the actual data to make it work more efficiently**

In [8]:
# Prefix to add in front of each QUOTE before embedding it
quote_prefix = {
    "MiniLM": "",
    "MPNet": "",
    "BGE-small": "",
    "E5-small": "passage: ",
}

# Prefix to add in front of the SITUATION you type in
situation_prefix = {
    "MiniLM": "",
    "MPNet": "",
    "BGE-small": "Represent this sentence for searching relevant passages: ",
    "E5-small": "query: ",
}

**step 6: embedding the dataset**

In [9]:
# This will store the embeddings for each model, e.g. quote_embeddings["MiniLM"]
quote_embeddings = {}

for nickname, model in models.items():
    print(f"Embedding all quotes with {nickname}...")

    # Add the right prefix (if any) to every quote's combined text
    texts_to_embed = quote_prefix[nickname] + df["combined_text"]

    # Ask the model to convert all that text into numbers
    # normalize_embeddings=True makes the similarity comparison in Step 7 simpler
    embeddings = model.encode(
        texts_to_embed.tolist(),
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    quote_embeddings[nickname] = embeddings

Embedding all quotes with MiniLM...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embedding all quotes with MPNet...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embedding all quotes with BGE-small...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embedding all quotes with E5-small...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

**step 7: this is a function to find the best anwser for the situation provided by us using cosine similarity**

In [10]:
def find_matching_quotes(situation, model_nickname, how_many=3):
    # Get the model we want to use
    model = models[model_nickname]

    # Add the right prefix for this model (empty string if none needed)
    text_to_embed = situation_prefix[model_nickname] + situation

    # Convert the situation into numbers
    situation_embedding = model.encode([text_to_embed], normalize_embeddings=True)

    # Compare the situation to every quote and get a similarity score for each
    all_quote_embeddings = quote_embeddings[model_nickname]
    similarity_scores = cosine_similarity(situation_embedding, all_quote_embeddings)[0]

    # Find the row numbers of the highest scores, largest first
    best_row_numbers = np.argsort(similarity_scores)[::-1][:how_many]

    # Build a small table with just the best quotes and their scores
    results = df.loc[best_row_numbers, ["quote", "author"]].copy()
    results["similarity_score"] = similarity_scores[best_row_numbers]

    return results.reset_index(drop=True)

In [11]:
test_situations = [
    "starting a new job and feeling nervous",
    "going through a breakup and trying to move on",
    "losing a close friend and grieving",
    "feeling stuck and unmotivated at work",
    "celebrating a hard-won achievement",
    "doubting yourself before a big decision",
]

**step 8: comparing models on each situation**

In [12]:
def compare_models_on_situation(situation, how_many=3):
    print("SITUATION:", situation)
    print("=" * 70)

    for nickname in models:
        print(f"\n{nickname}:")
        matches = find_matching_quotes(situation, nickname, how_many)
        for i in range(len(matches)):
            quote = matches.loc[i, "quote"]
            author = matches.loc[i, "author"]
            score = matches.loc[i, "similarity_score"]
            print(f'  {score:.3f}  "{quote}" — {author}')

In [13]:
for situation in test_situations:
    compare_models_on_situation(situation)
    print("\n" + "-" * 70 + "\n")

SITUATION: starting a new job and feeling nervous

MiniLM:
  0.402  "“No matter how careful you are, there's going to be the sense you missed something, the collapsed feeling under your skin that you didn't experience it all. There's that fallen heart feeling that you rushed right through the moments where you should've been paying attention.Well, get used to that feeling. That's how your whole life will feel some day.This is all practice.”" — Chuck Palahniuk,
  0.335  "“Laugh, even when you feel too sick or too worn out or tired. Smile, even when you're trying not to cry and the tears are blurring your vision. Sing, even when people stare at you and tell you your voice is crappy. Trust, even when your heart begs you not to. Twirl, even when your mind makes no sense of what you see. Frolick, even when you are made fun of. Kiss, even when others are watching. Sleep, even when you're afraid of what the dreams might bring. Run, even when it feels like you can't run any more.And, always, r

## ***after going through the results my conclusion says that miniLM is giving the most accurate result which are closest to the situation i have given.***